# Small effective-interaction simulation in a periodic box

This toy example demonstrates distinct bead types, a synthetic symmetric interaction matrix, periodic simulation, and a contact map. Each chain contains one bead, so this is a system of independent particles rather than bonded polymers. It uses the CPU Reference platform and a short run by default. Increase the parameters only after checking the small example.

Install this fork from the `update_v_0.2` branch with its test extra (`pip install -e ".[test]"`) in an environment that also provides Jupyter. This notebook creates synthetic data and writes results under `output/effective_example`. It does not use the excluded rheology experiments. The contact map is an illustration of the workflow, not an equilibrated research result.

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from chromdyn import ChromatinDynamics, TopologyGenerator, HiCManager

N_PARTICLES = 32
BOX_LENGTH = 12.0
RELAXATION_STEPS = 200
PRODUCTION_STEPS = 1000
REPORT_INTERVAL = 10
PLATFORM = "Reference"  # CPU; change to "CUDA" for larger runs if available.
SEED = 2026
OUTPUT_DIR = Path("output/effective_example")
rng = np.random.default_rng(SEED)

## Synthetic particles and interactions

The random matrix is symmetrized before use, with a zero diagonal. A deterministic NumPy seed fixes the matrix and initial positions; stochastic simulation trajectories can vary across runs and platforms. The periodic box has no additional spatial confinement.

In [ ]:
generator = TopologyGenerator()
generator.gen_top(chain_lens=[1] * N_PARTICLES, types="unique")
raw_matrix = rng.uniform(0.0, 0.2, (N_PARTICLES, N_PARTICLES))
interaction_matrix = -(raw_matrix + raw_matrix.T) / 2
np.fill_diagonal(interaction_matrix, 0.0)
initial_positions = rng.uniform(-BOX_LENGTH / 2, BOX_LENGTH / 2, (N_PARTICLES, 3))

sim = ChromatinDynamics(
    generator.topology, name="effective_example", platform_name=PLATFORM,
    output_dir=str(OUTPUT_DIR), console_stream=False, mass=1.0,
)
sim.force_field_manager.add_self_avoidance(Ecut=5.0, k=20.0, r=1.0)
sim.force_field_manager.add_type_to_type_interaction(
    interaction_matrix=interaction_matrix, type_labels=generator.atom_types,
    mu=3.22, rc=1.78,
)

## Short periodic simulation

The default run records 100 production frames. Relaxation is deliberately short for a runnable demonstration; choose relaxation and sampling based on the scientific question for production work.

In [ ]:
sim.simulation_setup(
    init_struct=initial_positions, integrator="langevin",
    temperature=120.0, timestep=0.002, friction=1.0,
    save_pos=True, save_energy=False, stability_check=False,
    pos_report_interval=REPORT_INTERVAL,
    PBC=True,
    box_vectors=((BOX_LENGTH, 0, 0), (0, BOX_LENGTH, 0), (0, 0, BOX_LENGTH)),
)
sim.run(RELAXATION_STEPS, verbose=False, report=False)
sim.run(PRODUCTION_STEPS, verbose=False, report=True)
trajectory_file = sim.reporters["position"].filename
sim.save_reports()
print(f"Trajectory saved to {trajectory_file}")

## Contact map

This call uses the ordinary within-frame contact definition, with `mu` and `rc` matching the interaction setup. It does not automatically include nearest periodic images. Use the explicit PBC contact-analysis API when that is the intended observable.

In [ ]:
hic = HiCManager().gen_hic_from_cndb(
    traj_file=trajectory_file, mu=3.22, rc=1.78, p=None,
    platform="CPU", parallel=False,
)
assert hic.shape == (N_PARTICLES, N_PARTICLES)
assert np.all(np.isfinite(hic))
np.testing.assert_allclose(hic, hic.T)
fig, ax = plt.subplots(figsize=(5, 4))
image = ax.imshow(np.maximum(hic, 1e-8), cmap="Reds", norm=LogNorm())
fig.colorbar(image, ax=ax, label="Contact weight")
ax.set(xlabel="Bead", ylabel="Bead", title="Toy effective-interaction contact map")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "contact_map.png", dpi=140)
plt.show()